[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/04-recall-tuning/05-detecting_phrases_in_free_text.ipynb)

In [1]:
# !pip install mbox

# Detecting Phrases in Free Text

`01-understanding_match_modes.ipynb` introduced `DETECT` as the mirror image of `COMPLETE`, a short indexed value found inside a longer query, and moved on. This notebook stays on `DETECT` specifically, because it's the mode behind a genuinely common, practical problem: a chatbot, a support inbox, a moderation queue, anywhere you have a long, messy piece of free text and need to know whether it mentions one of a small set of things you already care about.

In this notebook you will:

1. Confirm the basic shape: a short indexed phrase found inside a long query
2. Measure how much typo tolerance `DETECT` actually has, it is not literal substring search
3. See that it stays conservative about partial phrase coverage, one matching word out of two is not enough
4. Stress-test it against genuinely unrelated text and confirm it doesn't fire on noise
5. Detect more than one phrase in the same message
6. Set a real quality threshold with `min_qualities`
7. Find the one thing `DETECT` cannot do on its own, and see what has to close that gap

## Setting up

Four short phrases, the kind of thing a support bot would want to route on, loaded from `datasets/topics.csv`. Each is indexed as its own row, `DETECT` will be asked, for a given message, which of these four rows is mentioned inside it.

In [2]:
import pandas as pd
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallMode

topics = pd.read_csv("datasets/topics.csv")

index = TableIndexer.create_index(topics, index_columns=["phrase"], tmp_dir="tmp_index")
topics

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,phrase
0,refund
1,password reset
2,cancel subscription
3,shipping delay


## 1. The basic case

A full sentence, with one of the four phrases embedded in it somewhere.

In [3]:
result = index.match(
    phrase="I want a refund for my last order",
    modes={"phrase": TableRecallMode.DETECT},
    include_field_scores=True
)
result[["phrase_candidate", "phrase_score"]]

,phrase_candidate,phrase_score
0,refund,87


`refund` found, scored high, exactly the direction `COMPLETE` cannot do, the indexed value is far shorter than the query.

## 2. How much typo tolerance does `DETECT` actually have?

`DETECT` is not literal substring search, it's a fuzzy search for the best-aligned window inside the query, the same family of comparison as `APPROX`, just applied to a fragment instead of the whole string. Run the same four topics against a clean mention and a badly typo'd one.

In [4]:
test_cases = [
    ("I want a refund for my last order", "clean"),
    ("can I get a refnud for this please", "typo'd"),
    ("how do I reset my password", "clean"),
    ("i forgot my apssword and need to reset it", "typo'd"),
    ("please cancel subscription asap", "clean"),
    ("i need too cancell my subscrption", "typo'd"),
    ("my package shipping is delayed", "reworded"),
]

rows = []
for query, kind in test_cases:
    r = index.match(phrase=query, modes={"phrase": TableRecallMode.DETECT}, include_field_scores=True)
    rows.append((query, kind, r.iloc[0]["phrase_candidate"], int(r.iloc[0]["phrase_score"])))

pd.DataFrame(rows, columns=["query", "kind", "matched_phrase", "score"])

,query,kind,matched_phrase,score
0,I want a refund for my last order,clean,refund,87
1,can I get a refnud for this please,typo'd,refund,49
2,how do I reset my password,clean,password reset,92
3,i forgot my apssword and need to reset it,typo'd,password reset,71
4,please cancel subscription asap,clean,cancel subscription,91
5,i need too cancell my subscrption,typo'd,cancel subscription,70
6,my package shipping is delayed,reworded,shipping delay,94


Scores drop noticeably on the typo'd versions, roughly `85-94` clean versus `49-71` typo'd, but every single one still resolves to the right phrase. Notice the last row too: `"my package shipping is delayed"` still finds `shipping delay` even though the words aren't contiguous in the query, this is a fuzzy window search, not a literal substring check.

## 3. It stays conservative about partial coverage

`cancel subscription` and `password reset` are two-word phrases. A message containing only one of the two words, not the whole phrase, shouldn't count.

In [5]:
partial_cases = [
    "please cancel my gym membership",       # "cancel" without "subscription"
    "I need to change my password please",   # "password" without "reset"
    "the shipping was fine, no complaints",  # "shipping" without "delay"
]

for query in partial_cases:
    r = index.match(phrase=query, modes={"phrase": TableRecallMode.DETECT}, include_field_scores=True)
    found = r.iloc[0]["index_row"] != -1
    print(f"{query!r:55s} -> found: {found}")

'please cancel my gym membership'                       -> found: False
'I need to change my password please'                   -> found: False
'the shipping was fine, no complaints'                  -> found: False


None of them match. A single overlapping word out of a multi-word phrase isn't enough for `DETECT` to call it a hit, which matters a great deal for a chatbot: a message that happens to contain the word "password" for an unrelated reason shouldn't get routed to password-reset handling.

## 4. Stress test: does it fire on genuinely unrelated text?

Ten messages, none of which have anything to do with the four topics.

In [6]:
off_topic = [
    "hey do you have this in a different color",
    "what are your business hours on weekends",
    "can you tell me more about the premium plan features",
    "my account seems locked, what happened",
    "is there a mobile app for this service",
    "thank you so much for your help today",
    "I would like to speak to a human agent please",
    "the website keeps crashing when i click submit",
    "do you offer student discounts",
    "whats the difference between the basic and pro tier",
]

scores = []
for query in off_topic:
    r = index.match(phrase=query, modes={"phrase": TableRecallMode.DETECT}, include_field_scores=True)
    scores.append(int(r.iloc[0]["phrase_score"]))

print(f"Scores across {len(off_topic)} unrelated messages: {scores}")

Scores across 10 unrelated messages: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Every single one comes back `0`. Across this test set, `DETECT` didn't manufacture a single false positive, unrelated text stays unrelated.

## 5. Detecting more than one phrase at once

Real messages aren't always about exactly one thing. Ask for more than one result and see what a message touching two topics returns.

In [7]:
result = index.match(
    phrase="I want a refund because of the shipping delay on my order",
    modes={"phrase": TableRecallMode.DETECT},
    max_results=4,
    include_field_scores=True
)
result[["phrase_candidate", "phrase_score"]]

,phrase_candidate,phrase_score
0,shipping delay,84
1,refund,83


Both `shipping delay` and `refund` come back, a chatbot reading this result can address, or route to, both, rather than being forced to pick just one.

## 6. Setting a real threshold

Eyeballing scores is fine for exploration, a real system needs an actual cutoff. `min_qualities` enforces one per field, anything scoring below it is treated as not found at all, not just ranked lower.

In [8]:
for threshold in [None, 50]:
    kwargs = dict(phrase="can I get a refnud for this please", modes={"phrase": TableRecallMode.DETECT}, include_field_scores=True)
    if threshold is not None:
        kwargs["min_qualities"] = {"phrase": threshold}
    r = index.match(**kwargs)
    print(f"min_qualities={threshold} -> {r.iloc[0]['phrase_candidate']!r} at {r.iloc[0]['phrase_score']}")

min_qualities=None -> 'refund' at 49
min_qualities=50 -> '' at 0


The typo'd `refund` mention scored `49`, a `min_qualities` of `50` is enough to reject it entirely. That's close, on purpose: section 2 showed the worst typo case landing at `49`, right at that edge. Picking a threshold here is a real tradeoff between catching typos and rejecting noise, not a formality, and it should be set from your own traffic, not copied from this notebook.

## 7. What `DETECT` cannot do on its own

Every test so far mangled the *spelling* of a phrase. None of them changed the *words*. Try genuine synonyms instead, a customer who never says "refund" at all.

In [9]:
synonym_cases = [
    "I want my money back for this order",
    "can I get reimbursed for this purchase",
    "i forgot my login credentials",
    "please terminate my membership",
]

for query in synonym_cases:
    r = index.match(phrase=query, modes={"phrase": TableRecallMode.DETECT}, include_field_scores=True)
    print(f"{query!r:45s} -> score {int(r.iloc[0]['phrase_score'])}")

'I want my money back for this order'         -> score 0
'can I get reimbursed for this purchase'      -> score 0
'i forgot my login credentials'               -> score 0
'please terminate my membership'              -> score 0


Zero, every time. This is the honest limit of `DETECT` by itself: it tolerates *misspelling* a phrase gracefully, section 2 showed that clearly, but it has no way to know that "money back" and "refund" are the same request. That gap isn't a `DETECT` weakness to fix with a different mode, `APPROX`, `EXACT`, and `COMPLETE` all have the same blind spot, it's a vocabulary problem, and it needs a vocabulary, an alias for each phrase covering the ways people actually phrase the same request. `06-agentic-ai/09` picks up exactly here.

## Practical notes

**`DETECT` degrades gracefully on spelling, not on wording.** Use it when you're worried about typos in how someone spells your trigger phrase. Don't expect it to bridge a genuine synonym, that's a different problem with a different fix.

**Multi-word phrases need real coverage, not fragment overlap.** A phrase like `cancel subscription` won't fire on a message containing only `cancel` or only `subscription`. That's a safety property worth keeping in mind when choosing how many words to put in a trigger phrase, more words means more specificity, at the cost of needing more of the phrase to actually appear.

**Set `min_qualities` from real examples, not intuition.** Section 6's threshold of `50` was chosen by looking directly at where a real typo'd case landed. Do the same with a sample of your own traffic before picking a number.

**`DETECT` finds one phrase; a real pipeline should ask about every phrase you care about.** Section 5 asked for `max_results=4` against a single query on purpose, a message is not obligated to be about only one thing.

## Next steps

- **`06-agentic-ai/09-topic_routing_with_llm_enriched_aliases.ipynb`** - close the synonym gap from section 7 with LLM-generated, cached aliases, and route a chatbot message on the result